# Load datasets

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
import scanpy as sc
import pandas as pd
import numpy as np
from pathlib import Path  # <--- this fixes the NameError

In [3]:
RAW_DIR = Path(
    r"Q:\brca-scrnaseq-analysis\brca-scrnaseq-analysis-main\brca-scrnaseq-analysis-main\Data\Raw"
)

adata1 = sc.read_h5ad(
    RAW_DIR / "GSE114725_raw.h5ad"
)

adata2 = sc.read_h5ad(
    RAW_DIR / "GSE176078_raw.h5ad"
)

print(adata1)
print(adata2)

AnnData object with n_obs × n_vars = 47016 × 14875
    obs: 'patient', 'tissue', 'replicate', 'cluster'
AnnData object with n_obs × n_vars = 100064 × 29733
    obs: 'Unnamed: 0', 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mito', 'subtype', 'celltype_subset', 'celltype_minor', 'celltype_major', 'dataset'


# Calculate QC metrics

In [4]:
for adata in [adata1, adata2]:
    adata.var["mt"] = adata.var_names.str.startswith("MT-")
    sc.pp.calculate_qc_metrics(
        adata,
        qc_vars=["mt"],
        percent_top=None,
        log1p=False,
        inplace=True
    )

print(adata1.obs[["n_genes_by_counts", "total_counts", "pct_counts_mt"]].head())
print(adata2.obs[["n_genes_by_counts", "total_counts", "pct_counts_mt"]].head())

         n_genes_by_counts  total_counts  pct_counts_mt
cell_id                                                
246                    240         401.0       1.246883
260                    509         946.0       6.976744
346                    443         839.0       5.959476
188                    369         541.0       5.545287
33                     275         407.0       3.439803
                          n_genes_by_counts  total_counts  pct_counts_mt
CID3586_AAGACCTCAGCATGAG               1689        4581.0       1.506221
CID3586_AAGGTTCGTAGTACCT                779        1726.0       5.793743
CID3586_ACCAGTAGTTGTGGCC                514        1229.0       1.383238
CID3586_ACCCACTAGATGTCGG                609        1352.0       1.923077
CID3586_ACTGATGGTCAACTGT                807        1711.0      13.325541


In [26]:
from pathlib import Path
import matplotlib.pyplot as plt

FIGURE_DIR = Path("figures") / "phase1_qc"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print(FIGURE_DIR.resolve())

\\qub-stu-cfs1\stuhome1\2020\2\40322022\brca-scrnaseq-analysis\brca-scrnaseq-analysis-main\brca-scrnaseq-analysis-main\Notebooks\figures\phase1_qc


# Visulaise QC Distributions

In [35]:
# GSE114725 violin plot
sc.pl.violin(
    adata1,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
    show=False
)

plt.savefig(
    FIGURE_DIR / "GSE114725_before_filtering_violin.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# GSE176078 violin plot
sc.pl.violin(
    adata2,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
    show=False
)

plt.savefig(
    FIGURE_DIR / "GSE176078_before_filtering_violin.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

In [28]:
# GSE114725 scatter plot

sc.pl.scatter(
    adata1,
    x="total_counts",
    y="n_genes_by_counts",
    color="pct_counts_mt",
    show=False
)

plt.savefig(
    FIGURE_DIR / "GSE114725_counts_vs_genes_scatter.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# GSE176078 scatter plot

sc.pl.scatter(
    adata2,
    x="total_counts",
    y="n_genes_by_counts",
    color="pct_counts_mt",
    show=False
)

plt.savefig(
    FIGURE_DIR / "GSE176078_counts_vs_genes_scatter.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

Filter low quility cells

In [7]:
print("Before QC filtering:")
print("GSE114725:", adata1.n_obs, "cells,", adata1.n_vars, "genes")
print("GSE176078:", adata2.n_obs, "cells,", adata2.n_vars, "genes")

Before QC filtering:
GSE114725: 47016 cells, 14875 genes
GSE176078: 100064 cells, 29733 genes


In [8]:
adata1_qc = adata1[
    (adata1.obs["n_genes_by_counts"] > 200) &
    (adata1.obs["pct_counts_mt"] < 10)
].copy()

adata2_qc = adata2[
    (adata2.obs["n_genes_by_counts"] > 300) &
    (adata2.obs["pct_counts_mt"] < 10)
].copy()

print("After cell filtering:")
print("GSE114725:", adata1_qc.n_obs, "cells")
print("GSE176078:", adata2_qc.n_obs, "cells")

After cell filtering:
GSE114725: 36613 cells
GSE176078: 83033 cells


Filter low frequency genes

In [9]:
sc.pp.filter_genes(adata1_qc, min_cells=3)
sc.pp.filter_genes(adata2_qc, min_cells=3)

print("After gene filtering:")
print("GSE114725:", adata1_qc.n_obs, "cells,", adata1_qc.n_vars, "genes")
print("GSE176078:", adata2_qc.n_obs, "cells,", adata2_qc.n_vars, "genes")

After gene filtering:
GSE114725: 36613 cells, 14751 genes
GSE176078: 83033 cells, 27221 genes


#doublet detetction with Scrublet

In [10]:
import scrublet as scr

In [37]:
scrub = scr.Scrublet(adata.X)
doublet_scores, predicted_doublets = scrub.scrub_doublets(n_prin_comps=min(adata.shape[0], adata.shape[1], 15))

Preprocessing...


C:\Users\40322022\AppData\Roaming\Python\Python310\site-packages\scrublet\helper_functions.py:239: RuntimeWarning: invalid value encountered in log
  gLog = lambda input: np.log(input[1] * np.exp(-input[0]) + input[2])
C:\Users\40322022\AppData\Roaming\Python\Python310\site-packages\scrublet\helper_functions.py:252: RuntimeWarning: invalid value encountered in sqrt
  CV_input = np.sqrt(b);


Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.52
Detected doublet rate = 0.4%
Estimated detectable doublet fraction = 63.2%
Overall doublet rate:
	Expected   = 10.0%
	Estimated  = 0.7%
Elapsed time: 181.2 seconds


In [41]:
def run_scrublet(adata, dataset_name):
    print(f"Running Scrublet for {dataset_name}")

    before = adata.n_obs

    n_pcs = min(adata.shape[0], adata.shape[1], 15) - 1
    print(f"Using {n_pcs} PCs for Scrublet")

    scrub = scr.Scrublet(adata.X)

    doublet_scores, predicted_doublets = scrub.scrub_doublets(
        n_prin_comps=n_pcs
    )

    adata.obs["doublet_score"] = doublet_scores
    adata.obs["predicted_doublet"] = predicted_doublets

    sc.pl.violin(
        adata,
        ["doublet_score"],
        jitter=0.4,
        show=False
    )

    plt.savefig(
        FIGURE_DIR / f"{dataset_name}_scrublet_doublet_scores.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    adata = adata[~adata.obs["predicted_doublet"]].copy()

    print(f"Cells before: {before}")
    print(f"Cells after: {adata.n_obs}")
    print(f"Doublets removed: {before - adata.n_obs}")

    return adata

In [42]:
adata1_qc = run_scrublet(adata1_qc, "GSE114725")

Running Scrublet for GSE114725
Using 14 PCs for Scrublet
Preprocessing...


C:\Users\40322022\AppData\Roaming\Python\Python310\site-packages\scrublet\helper_functions.py:252: RuntimeWarning: invalid value encountered in sqrt
  CV_input = np.sqrt(b);


Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.61
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 3.6%
Overall doublet rate:
	Expected   = 10.0%
	Estimated  = 0.2%
Elapsed time: 26.7 seconds
Cells before: 36610
Cells after: 36607
Doublets removed: 3


In [43]:
adata2_qc = run_scrublet(adata2_qc, "GSE176078")

Running Scrublet for GSE176078
Using 14 PCs for Scrublet
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.67
Detected doublet rate = 0.1%
Estimated detectable doublet fraction = 3.9%
Overall doublet rate:
	Expected   = 10.0%
	Estimated  = 2.4%
Elapsed time: 103.3 seconds
Cells before: 83009
Cells after: 82931
Doublets removed: 78


# Normalisation

In [12]:
for adata in [adata1_qc, adata2_qc]:

    sc.pp.normalize_total(
        adata,
        target_sum=1e4
    )

    sc.pp.log1p(adata)

print("Normalisation complete")

Normalisation complete


Data were normalised using total-count normalisation to 10,000 counts per cell followed by log1p transformation. This standard Scanpy workflow reduces sequencing-depth differences between cells while stabilising variance for downstream analysis.

# Batch effect assessment and correction

Batch effects were assessed by visualising cells with UMAP before and after Harmony correction. For GSE114725, patient identity was treated as the batch variable. For GSE176078, `orig.ident` was used because it represents the sample/patient identifier. Harmony was used to reduce technical/sample-level effects while checking that biological structure was not completely removed.

In [13]:
import scanpy.external as sce

In [14]:
for adata in [adata1_qc, adata2_qc]:
    sc.pp.highly_variable_genes(
        adata,
        n_top_genes=2000,
        flavor="seurat"
    )

print("GSE114725 HVGs:", adata1_qc.var["highly_variable"].sum())
print("GSE176078 HVGs:", adata2_qc.var["highly_variable"].sum())

GSE114725 HVGs: 2000
GSE176078 HVGs: 2000


In [15]:
adata1_qc = adata1_qc[
    :,
    adata1_qc.var.highly_variable
].copy()

adata2_qc = adata2_qc[
    :,
    adata2_qc.var.highly_variable
].copy()

Scale + PCA + neighbours + UMAP

In [16]:
def run_pca_umap(adata):
    sc.pp.scale(adata, max_value=10)
    sc.tl.pca(adata, svd_solver="arpack")
    sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)
    sc.tl.umap(adata)
    return adata

adata1_qc = run_pca_umap(adata1_qc)
adata2_qc = run_pca_umap(adata2_qc)

C:\Program Files\Python310\lib\functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)
C:\Users\40322022\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Program Files\Python310\lib\functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)


In [17]:
# Run PCA, neighbours and UMAP before Harmony

def run_pca_umap(adata):
    sc.pp.scale(adata, max_value=10)

    sc.tl.pca(
        adata,
        svd_solver="arpack"
    )

    sc.pp.neighbors(
        adata,
        n_neighbors=15,
        n_pcs=30
    )

    sc.tl.umap(adata)

    return adata


adata1_qc = run_pca_umap(adata1_qc)
adata2_qc = run_pca_umap(adata2_qc)

print("PCA and pre-Harmony UMAP complete")

PCA and pre-Harmony UMAP complete


In [44]:
# GSE114725 before Harmony

sc.pl.umap(
    adata1_qc,
    color=["patient", "tissue"],
    title=[
        "GSE114725: patient before Harmony",
        "GSE114725: tissue before Harmony"
    ],
    show=False
)

plt.savefig(
    FIGURE_DIR / "GSE114725_before_harmony.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

In [45]:
# GSE176078 before Harmony

sc.pl.umap(
    adata2_qc,
    color=["orig.ident", "subtype", "celltype_major"],
    title=[
        "GSE176078: sample before Harmony",
        "GSE176078: subtype before Harmony",
        "GSE176078: cell type before Harmony"
    ],
    show=False
)

plt.savefig(
    FIGURE_DIR / "GSE176078_before_harmony.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

Harmony Batch correction

In [20]:
import scanpy.external as sce

In [21]:
# For GSE114725
sce.pp.harmony_integrate(
    adata1_qc,
    key="patient",   # or the batch key relevant for this dataset
    basis="X_pca"    # Harmony works on PCA space
)

# For GSE176078
sce.pp.harmony_integrate(
    adata2_qc,
    key="orig.ident",  # batch key for this dataset
    basis="X_pca"
)

2026-05-28 11:39:34,823 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
2026-05-28 11:39:39,571 - harmonypy - INFO - sklearn.KMeans initialization complete.
2026-05-28 11:39:39,726 - harmonypy - INFO - Iteration 1 of 10
2026-05-28 11:39:48,490 - harmonypy - INFO - Iteration 2 of 10
2026-05-28 11:39:57,485 - harmonypy - INFO - Converged after 2 iterations
2026-05-28 11:39:57,547 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
2026-05-28 11:40:03,368 - harmonypy - INFO - sklearn.KMeans initialization complete.
2026-05-28 11:40:03,692 - harmonypy - INFO - Iteration 1 of 10
2026-05-28 11:40:28,977 - harmonypy - INFO - Iteration 2 of 10
2026-05-28 11:40:54,029 - harmonypy - INFO - Iteration 3 of 10
2026-05-28 11:41:19,635 - harmonypy - INFO - Iteration 4 of 10
2026-05-28 11:41:45,826 - harmonypy - INFO - Iteration 5 of 10
2026-05-28 11:42:12,315 - harmonypy - INFO - Iteration 6 of 10
2026-05-28 11:42:37,790 - harmonypy - INFO - Iteration 7 of 

In [47]:
print(adata1_qc.obs.columns)
print(adata2_qc.obs.columns)

Index(['patient', 'tissue', 'replicate', 'cluster', 'n_genes_by_counts',
       'total_counts', 'total_counts_mt', 'pct_counts_mt', 'doublet_score',
       'predicted_doublet'],
      dtype='object')
Index(['Unnamed: 0', 'orig.ident', 'nCount_RNA', 'nFeature_RNA',
       'percent.mito', 'subtype', 'celltype_subset', 'celltype_minor',
       'celltype_major', 'dataset', 'n_genes_by_counts', 'total_counts',
       'total_counts_mt', 'pct_counts_mt', 'doublet_score',
       'predicted_doublet'],
      dtype='object')


In [48]:
sc.pl.umap(
    adata1_qc,
    color=["patient", "tissue"],
    title=[
        "GSE114725: patient after Harmony",
        "GSE114725: tissue after Harmony"
    ],
    show=False
)

plt.savefig(
    FIGURE_DIR / "GSE114725_after_harmony.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

In [49]:
sc.pl.umap(
    adata2_qc,
    color=["orig.ident", "subtype", "celltype_major"],
    title=[
        "GSE176078: sample after Harmony",
        "GSE176078: subtype after Harmony",
        "GSE176078: cell type after Harmony"
    ],
    show=False
)

plt.savefig(
    FIGURE_DIR / "GSE176078_after_harmony.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

#Testing

In [51]:
!pip install pytest

Defaulting to user installation because normal site-packages is not writeable

   ---------------------------------------- 0/3 [pluggy]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   ---------------------------------------- 3/3 [pytest]



  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [52]:
import pytest
import numpy as np
import pandas as pd
import scanpy as sc
from anndata import AnnData

In [54]:
import numpy as np
import pandas as pd
from anndata import AnnData


def test_low_gene_filter():
    """
    Cells with too few detected genes should be removed.
    """

    X = np.random.poisson(1, (4, 100))

    obs = pd.DataFrame({
        "n_genes_by_counts": [50, 150, 300, 500],
        "pct_counts_mt": [5, 5, 5, 5]
    })

    adata = AnnData(X, obs=obs)

    filtered = adata[
        adata.obs["n_genes_by_counts"] >= 200
    ]

    assert filtered.n_obs == 2


def test_mitochondrial_filter():
    """
    Cells with high mitochondrial percentage should be removed.
    """

    X = np.random.poisson(1, (4, 100))

    obs = pd.DataFrame({
        "n_genes_by_counts": [300, 300, 300, 300],
        "pct_counts_mt": [5, 10, 35, 40]
    })

    adata = AnnData(X, obs=obs)

    filtered = adata[
        adata.obs["pct_counts_mt"] < 20
    ]

    assert filtered.n_obs == 2


def test_scrublet_columns_exist():
    """
    Scrublet columns should exist in adata.obs.
    """

    X = np.random.poisson(1, (3, 100))

    adata = AnnData(X)

    adata.obs["doublet_score"] = [0.1, 0.8, 0.2]
    adata.obs["predicted_doublet"] = [False, True, False]

    assert "doublet_score" in adata.obs.columns
    assert "predicted_doublet" in adata.obs.columns
    assert adata.obs["predicted_doublet"].sum() == 1

In [60]:
!python -m pytest ../Tests/test_qc.py -v

============================= test session starts =============================
platform win32 -- Python 3.10.4, pytest-9.0.3, pluggy-1.6.0 -- C:\Program Files\Python310\python.exe
cachedir: .pytest_cache
rootdir: Q:\brca-scrnaseq-analysis\brca-scrnaseq-analysis-main\brca-scrnaseq-analysis-main
plugins: anyio-4.13.0
collecting ... collected 3 items

..\Tests\test_qc.py::test_low_gene_filter PASSED                         [ 33%]
..\Tests\test_qc.py::test_mitochondrial_filter PASSED                    [ 66%]
..\Tests\test_qc.py::test_scrublet_columns_exist PASSED                  [100%]

============================== warnings summary ===============================
Tests/test_qc.py::test_low_gene_filter
Tests/test_qc.py::test_mitochondrial_filter
  C:\Users\40322022\AppData\Roaming\Python\Python310\site-packages\anndata\_core\aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
    warnings.warn("Transforming to str index.", ImplicitModificationWarning)

-- Docs: ht

### Stretch task: multi-dataset pipeline structure

As a stretch objective, the Phase 1 workflow was structured to process both datasets using repeated functions and loops where possible. This allows the same QC, normalization, HVG selection, dimensionality reduction and plotting workflow to be applied to GSE114725 and GSE176078 with dataset-specific filtering thresholds where needed.

SCTransform/scran normalization and ambient RNA correction were not implemented at this stage because the priority was to complete a stable, reproducible Phase 1 QC pipeline.

In [62]:
from pathlib import Path

PROJECT_DIR = Path(
    r"Q:\brca-scrnaseq-analysis\brca-scrnaseq-analysis-main\brca-scrnaseq-analysis-main"
)

PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [63]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

adata1_qc.write(PROCESSED_DIR / "GSE114725_phase1_qc_harmony.h5ad")
adata2_qc.write(PROCESSED_DIR / "GSE176078_phase1_qc_harmony.h5ad")

RuntimeError: Can't decrement id ref count (unable to extend file properly)

In [64]:
print(adata1_qc)
print(adata2_qc)

AnnData object with n_obs × n_vars = 36607 × 2000
    obs: 'patient', 'tissue', 'replicate', 'cluster', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'doublet_score', 'predicted_doublet'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
    uns: 'log1p', 'hvg', 'pca', 'neighbors', 'umap', 'patient_colors', 'tissue_colors'
    obsm: 'X_pca', 'X_umap', 'X_pca_harmony'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'
AnnData object with n_obs × n_vars = 82931 × 2000
    obs: 'Unnamed: 0', 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mito', 'subtype', 'celltype_subset', 'celltype_minor', 'celltype_major', 'dataset', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'doublet_score', 'predicted_doublet'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cel

In [66]:
adata2_qc.write(
    PROCESSED_DIR / "GSE176078_phase1_qc_harmony_v2.h5ad"
)

print("Saved adata2 again")

RuntimeError: Can't decrement id ref count (unable to extend file properly)

In [ ]:
# having storage issues 

In [68]:
# Save compressed processed datasets

adata1_hvg = adata1_qc[:, adata1_qc.var['highly_variable']].copy()

adata2_hvg = adata2_qc[:, adata2_qc.var['highly_variable']].copy()


adata1_hvg.write(
    PROCESSED_DIR / "GSE114725_phase1_qc_harmony.h5ad",
    compression="gzip"
)

adata2_hvg.write(
    PROCESSED_DIR / "GSE176078_phase1_qc_harmony.h5ad",
    compression="gzip"
)

print("Compressed Phase 1 datasets saved successfully")

Compressed Phase 1 datasets saved successfully


## Phase 1 Processed Data

The Phase 1 QC- and Harmony-processed datasets have been saved in compressed `.h5ad` format:

- `GSE114725_phase1_qc_harmony.h5ad`
- `GSE176078_phase1_qc_harmony.h5ad`

### Notes:

- These files contain only highly variable genes (HVGs) and the QC-filtered cells.
- Compression (`gzip`) reduces file size significantly, making them easier to move between machines.
- These files are ready for **Phase 2: Clustering and Cell Type Annotation**.
- They do **not** include raw data; raw `.h5ad` files remain separate and should not be pushed to GitHub.
- Using compressed `.h5ad` ensures full compatibility with Scanpy for downstream analysis while saving disk space.